# Geopolitical-Exposure Portfolios

This exploratory Ukraine experiment converts the frozen exposure signals into investable, apples-to-apples long–short portfolios. It compares direct company disclosure with seed-excluded holdings-embedding expansion, characteristic-similarity expansion, and a fixed 50/50 direct-plus-embedding construction.

All selection uses information frozen before February 24, 2022. No API calls, WRDS queries, embedding refits, or weight optimization against Ukraine returns occur here. Because Ukraine results have already been inspected during model development, this notebook develops the architecture; it is not a pristine out-of-sample test.

In [ ]:
from pathlib import Path
import hashlib,json,sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
ROOT=Path.cwd()
if ROOT.name!='geopolitical_exposure': ROOT=ROOT/'geopolitical_exposure'
sys.path.insert(0,str(ROOT))
OUTPUT=ROOT/'exposure_portfolios'; FIGURES=OUTPUT/'figures'
OUTPUT.mkdir(exist_ok=True); FIGURES.mkdir(exist_ok=True)
from src.exposure_portfolios import CAR_COLUMNS,build_score_universe,select_members,evaluate_portfolios,contributor_audit
sns.set_theme(style='whitegrid',context='notebook')
N_PER_SIDE=25
print('Firms per side:',N_PER_SIDE)

## 1. Freeze and audit inputs

The primary model uses 2021Q4 all-S12 RS-L-Min embeddings. Event outcomes never enter firm selection. File hashes make the run auditable.

In [ ]:
required=[ROOT/'embedding_expansion_v2/directional_firm_channel_seeds.parquet',ROOT/'embedding_expansion_v2/embedding_propagated_scores.parquet',ROOT/'embedding_expansion_v2/characteristic_propagated_scores.parquet',ROOT/'event_data/market_model_cars.parquet',ROOT/'event_data/crsp_point_in_time_industry_2021q4.parquet',ROOT.parent/'remaining_benchmarks/return_comovement/characteristics/2021q4.parquet',ROOT.parent/'active_passive_extension/benchmarks/all_s12_control/2021q4/rs_l_min/asset_embeddings_10d.parquet']
audit=[]
for path in required:
    if not path.exists(): raise FileNotFoundError(path)
    audit.append({'path':str(path),'bytes':path.stat().st_size,'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
audit=pd.DataFrame(audit);audit.to_parquet(OUTPUT/'input_audit.parquet',index=False);display(audit[['path','bytes']])

## 2. Construct four matched-size portfolios

Each portfolio contains 25 long and 25 short firms. The direct-text portfolio ranks accepted positive and negative disclosures. Embedding and characteristic portfolios exclude every direct seed. The combined portfolio allocates approximately half of each side to direct text and half to nonseed embedding expansion. Zero text exposure is never interpreted as negative exposure.

In [ ]:
members=select_members(ROOT,count=N_PER_SIDE)
members.to_parquet(OUTPUT/'frozen_portfolio_members.parquet',index=False)
member_audit=members.groupby(['portfolio','side']).agg(names=('permco','size'),unique_names=('permco','nunique'),minimum_score=('selection_score','min'),maximum_score=('selection_score','max')).reset_index()
display(member_audit)
display(members.sort_values(['portfolio','side','selection_score'],ascending=[True,True,False]))

## 3. Equal-weighted and beta-neutral event returns

Equal-weighted portfolios are 100% long and 100% short. Beta-neutral weights are the closest feasible weights to equal weighting subject to the same long and short gross exposures, zero market beta, and a 15% name cap. Returns are market-model cumulative abnormal returns for CAR[0,1], CAR[0,5], and CAR[1,20].

In [ ]:
portfolio_results,weights=evaluate_portfolios(ROOT,members)
portfolio_results.to_parquet(OUTPUT/'portfolio_event_results.parquet',index=False)
weights.to_parquet(OUTPUT/'portfolio_weights_and_returns.parquet',index=False)
display(portfolio_results.sort_values(['weighting','car_0_5'],ascending=[True,False]))

## 4. Attribution and concentration audit

A portfolio result is not informative if it comes from one stock or an unintended industry bet. The following tables report member contributions and industry weights.

In [ ]:
attribution=weights.copy();attribution['car_0_5_contribution']=attribution.weight*attribution.car_0_5
attribution.to_parquet(OUTPUT/'car_0_5_attribution.parquet',index=False)
industry=(weights.groupby(['portfolio','weighting','side','sic2'],as_index=False).weight.sum().sort_values(['portfolio','weighting','side','weight'],ascending=[True,True,True,False]))
industry.to_parquet(OUTPUT/'industry_exposure.parquet',index=False)
display(attribution.sort_values('car_0_5_contribution',key=lambda x:x.abs(),ascending=False).head(30))
display(industry.groupby(['portfolio','weighting','side']).head(5))

## 5. Embedding quintile monotonicity

A useful exposure score should ideally produce an ordered response rather than relying only on two tails. The signed rank score contrasts proximity to positive seed channels with proximity to negative seed channels. Direct seeds remain excluded.

In [ ]:
embedding,_=build_score_universe(ROOT)
cars=pd.read_parquet(ROOT/'event_data/market_model_cars.parquet')
q=embedding.loc[~embedding.is_any_seed].merge(cars,on='permco',how='inner',validate='one_to_one').copy()
q['score_quintile']=pd.qcut(q.signed_rank_score,5,labels=[1,2,3,4,5])
quintiles=q.groupby('score_quintile',observed=True)[CAR_COLUMNS].agg(['mean','count']).reset_index()
quintiles.to_parquet(OUTPUT/'embedding_score_quintiles.parquet',index=False)
display(quintiles)

## 6. Reveal the seed firms behind every embedding selection

For each embedding-only portfolio member, this audit reports its three closest same-direction text seeds in the underlying 10-dimensional asset space. Candidates supported only by economically incoherent neighbors should not be interpreted as discovered exposures.

In [ ]:
contributors=contributor_audit(ROOT,members,neighbors=3)
contributors.to_parquet(OUTPUT/'embedding_candidate_seed_contributors.parquet',index=False)
display(contributors)

## 7. Random-portfolio reference distribution

The empirical portfolios are compared with 1,000 randomly signed, matched-size nonseed portfolios. This tests whether the observed spread is unusual relative to arbitrary firm selection; it does not replace a held-out event.

In [ ]:
rng=np.random.default_rng(20220224)
cars=pd.read_parquet(ROOT/'event_data/market_model_cars.parquet').dropna(subset=CAR_COLUMNS)
seed_ids=set(pd.read_parquet(ROOT/'embedding_expansion_v2/directional_firm_channel_seeds.parquet').firm_id.astype(int))
pool=cars.loc[~cars.permco.isin(seed_ids)].copy(); placebo=[]
for repetition in range(1000):
    draw=pool.sample(2*N_PER_SIDE,replace=False,random_state=int(rng.integers(2**31-1)))
    for outcome in CAR_COLUMNS:
        placebo.append({'repetition':repetition,'outcome':outcome,'spread':draw.iloc[:N_PER_SIDE][outcome].mean()-draw.iloc[N_PER_SIDE:][outcome].mean()})
placebo=pd.DataFrame(placebo);placebo.to_parquet(OUTPUT/'random_portfolio_placebos_1000.parquet',index=False)
actual=portfolio_results.query("weighting=='equal_weight'").copy()
pvalues=[]
for row in actual.itertuples():
    for outcome in CAR_COLUMNS:
        null=placebo.loc[placebo.outcome.eq(outcome),'spread'];value=getattr(row,outcome)
        pvalues.append({'portfolio':row.portfolio,'outcome':outcome,'car':value,'two_sided_random_pvalue':(1+(null.abs()>=abs(value)).sum())/(1+len(null))})
placebo_test=pd.DataFrame(pvalues);placebo_test.to_parquet(OUTPUT/'random_portfolio_test.parquet',index=False);display(placebo_test)

## 8. Visual comparison

The figure compares identical portfolio constructions across three event windows. Data labels show cumulative abnormal returns in percentage points.

In [ ]:
plot=portfolio_results.melt(id_vars=['portfolio','weighting'],value_vars=CAR_COLUMNS,var_name='window',value_name='car')
g=sns.catplot(data=plot,x='portfolio',y='car',hue='weighting',col='window',kind='bar',height=5,aspect=1.05,palette=['#4472C4','#ED7D31'])
for ax in g.axes.flat:
    ax.axhline(0,color='black',lw=1);ax.tick_params(axis='x',rotation=30);ax.set_xlabel('')
    for container in ax.containers: ax.bar_label(container,labels=[f'{v*100:.2f}%' for v in container.datavalues],padding=2,fontsize=8)
g.set_axis_labels('', 'Cumulative abnormal return');g.fig.suptitle('Geopolitical-exposure portfolio results',y=1.03)
plt.savefig(FIGURES/'portfolio_comparison.png',dpi=200,bbox_inches='tight');plt.show()

## 9. Interpretation and next-event protocol

Ukraine is a development event. A successful result here establishes feasibility, not generalization. Before examining another event's returns, freeze: event-selection criteria; filing cutoff; exposure channels; extraction prompt; evidence rules; positive/negative relationship classifier; universe; number of names; weighting and neutrality rules; event windows; baselines; and multiple-testing policy.

The eventual claim should be that embeddings improve retrieval of independently verified indirect exposures—not merely that one retrospectively designed portfolio earned a positive return.

In [ ]:
inventory=[]
for path in sorted(OUTPUT.glob('*.parquet')):
    inventory.append({'artifact':path.name,'rows':len(pd.read_parquet(path)),'bytes':path.stat().st_size,'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
inventory=pd.DataFrame(inventory)
manifest={'stage':'exploratory_geopolitical_exposure_portfolios','event':'Russia invasion of Ukraine','event_date':'2022-02-24','development_event':True,'names_per_side':N_PER_SIDE,'artifacts':inventory.to_dict('records')}
(OUTPUT/'manifest.json').write_text(json.dumps(manifest,indent=2));display(inventory)
print('Complete. Interpret Ukraine as architecture development and freeze all rules before a second event.')